# NB3 — MobileViT Backbone Validation — TopoDistil++

Sanity/validation notebook for the hybrid-architecture branch (Section "Recommended new structure", `TopoDistil_bulletproof.md`). Purpose: confirm a MobileViT-family backbone implements correctly, fits comfortably on a Kaggle T4, and trains at a reasonable speed — **before** LIFE (topology gating) is added on top of it in NB4.

This notebook does **not** train with topology supervision. It only needs NB1's checkpoint for the fixed split indices (for an apples-to-apples comparison against NB2's RepViT run) — it does not touch `A_gauss_maps.h5` or the persistence diagrams at all.

```
NB1 (topology)         NB2 (RepViT, done)
      |                        |
      |                        v
      +----------------> NB3 (this notebook)
                          MobileViT sanity check
                                |
                                v
                          NB4: MobileViT + LIFE
```


## 0. Setup

In [1]:
!pip install -q timm thop fvcore


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 5.2 MB/s eta 0:00:00


In [2]:
import os, json, time, random, math
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F_
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm.auto import tqdm

import timm

try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU detected. Shape/param/FLOPs checks below will still run, but the\n"
          "GPU-memory and training-speed sections need a Kaggle GPU runtime (Settings > Accelerator > GPU T4 x2)\n"
          "to give numbers that mean anything for the T4 budget this notebook exists to check.")


Device: cuda


## 1. Implementation assumptions

1. **Scope.** NB3 only validates the *backbone*. No `TopoGate`, no `ResidualProjector`, no
   `L_attn`, no topology data loaded. That machinery is already implemented and unit-tested in
   NB2 (`TopoDistilModel` / `TopoGate` / `build_A_proj`) and is reused as-is in NB4 once this
   notebook confirms the backbone is sound — no need to re-derive it here.
2. **Backbone size-matching.** `mobilevit_s` (~4.9M params at 96×96) is the default candidate
   because it is the closest size match to NB2's `repvit_m1.dist_in1k` (~4.7M params), which
   keeps the RepViT-vs-MobileViT comparison in NB5 a fair "similar-capacity efficient backbone"
   comparison rather than confounding architecture family with model size. `mobilevit_xs` and
   `mobilevit_xxs` are kept as automatic fallbacks (Section 7) purely for T4 memory headroom;
   if a fallback is used, NB5's cross-backbone comparison should note the parameter-count
   mismatch explicitly.
3. **CNN/Transformer injection boundary.** Per the hybrid architecture in
   `TopoDistil_bulletproof.md` ("I wouldn't put the topology gate at the very beginning of
   MobileViT... CNN stem -> local feature map -> topology gate -> tokenization -> Transformer
   block"), the gate must sit on the feature map produced by the last pure-CNN
   (`BottleneckBlock`) stage, immediately before the first attention (`MobileVitBlock`) stage.
   Section 4 below detects this boundary programmatically instead of hand-picking a layer name
   (unlike NB2's `pick_injection_layer`, which just takes the midpoint of a pure-CNN backbone —
   there is no CNN/Transformer boundary to respect there). The detected layer name is saved to
   `config_used.json` so NB4 loads it directly rather than re-deriving it.
4. **"Baseline training" here is a sanity run, not the real baseline.** NB4's experimental
   matrix trains `MobileViT baseline` with 3 seeds over the full epoch budget as one of its four
   configurations — that is the number that goes in the results table. The single short run in
   Section 9 here exists only to prove the training loop is bug-free and to measure per-epoch
   wall-clock time on this GPU, extrapolated to the full budget, so the T4 time budget for NB4
   can be planned before committing to it.
5. **Pretrained weights.** Matches NB2's convention: the model actually trained/timed in
   Sections 7 and 9 uses ImageNet-pretrained weights (`pretrained=True`). The architecture-only
   comparisons in Sections 4–6 (shape check, param count, FLOPs across candidate sizes) use
   `pretrained=False` since weight values don't affect shape, param count, or FLOPs, and it
   avoids downloading three sets of weights just to compare architectures.


## 2. Config

In [ ]:
CONFIG = {
    "NB1_DIR": None,              # auto-discovered in Section 3 by searching for splits.json -- this placeholder is never read
    "PCAM_DIR": "/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon",

    "BACKBONE_CANDIDATES": ["mobilevit_s", "mobilevit_xs", "mobilevit_xxs"],  # tried in order
    "BACKBONE": None,              # filled in by the GPU-memory fallback probe (Section 7)
    "INJECTION_LAYER": None,       # filled in by the CNN/Transformer boundary detector (Section 4)
    "IMG_SIZE": 96,                # native PCam patch size, same as NB1/NB2

    "BATCH_SIZE": 32,              # matches NB2, so GPU-memory/timing numbers are comparable
    "LR": 1e-3,                    # peak LR for the freshly-initialized classifier head
    "LR_BACKBONE_MULT": 0.1,       # pretrained backbone body gets LR * this (discriminative LR, matches NB2/NB4 fix)
    "WARMUP_EPOCHS": 1,            # linear warmup, then cosine decay over the rest of SANITY_EPOCHS
    "MIN_LR_FRAC": 0.05,           # cosine decay floor, as a fraction of peak LR
    "SANITY_EPOCHS": 2,            # quick correctness/speed check, NOT the real NB4 baseline
    "FULL_EPOCHS_REFERENCE": 12,   # NB2/NB4's real epoch budget, used only to extrapolate timing
    "SANITY_TRAIN_SUBSET": 3000,   # cap sanity-run train set size for speed; set None to use all of NB1's train_idx

    "VAL_SUBSET": 2000,
    "TEST_SUBSET": 2000,

    "SEED": 0,

    "OUT_DIR": "/kaggle/working/mobilevit_baseline",
}
os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["SEED"])
CONFIG


## 3. Load NB1 checkpoint

Only `splits.json` is needed here (fixed train/val/test indices), so this notebook trains and evaluates on **exactly the same patches** as NB2's RepViT runs. `A_gauss_maps.h5` and the persistence diagrams are intentionally not loaded -- this notebook has nothing to do with them.

NB1's output directory is auto-discovered by searching `/kaggle/input` for `splits.json`, rather than a hardcoded path -- Kaggle's "Add Input" mounts a notebook's output under a path that includes *your* username/slug, not a fixed string, so a hardcoded guess breaks for anyone except whoever originally wrote it. (An earlier version of this notebook had a second, hardcoded-path cell here that ran *before* this discovery cell and would raise before the working fallback ever got a chance to execute -- removed.)

In [4]:
from pathlib import Path
import json
import numpy as np

# ============================================================
# 1. FIND NB1 TOPOLOGY OUTPUT AUTOMATICALLY
# ============================================================

def find_nb1_checkpoint(search_root="/kaggle/input"):
    """
    Find NB1's topology_checkpoint directory by locating splits.json.
    Searches all Kaggle inputs, so it does not depend on the
    displayed notebook/output name or the owning account's slug.
    """
    root = Path(search_root)

    if not root.exists():
        raise FileNotFoundError(f"Kaggle input directory not found: {root}")

    matches = list(root.rglob("splits.json"))

    # Prefer splits.json located inside topology_checkpoint
    for f in matches:
        if f.parent.name.lower() == "topology_checkpoint":
            return f.parent

    # Fallback: if there is only one splits.json, use it
    if len(matches) == 1:
        return matches[0].parent

    return None


nb1_dir = find_nb1_checkpoint()

if nb1_dir is None:
    print("ERROR: Could not find NB1 topology checkpoint.")
    print("\nAll splits.json files found under /kaggle/input:")
    for f in Path("/kaggle/input").rglob("splits.json"):
        print("  ", f)
    raise FileNotFoundError(
        "\nNB1 output was not found. "
        "Make sure the NB1 notebook output is attached "
        "using Kaggle \u2192 Add Input \u2192 Notebook Output."
    )

CONFIG["NB1_DIR"] = str(nb1_dir)

print("\u2713 NB1 checkpoint found:")
print(" ", CONFIG["NB1_DIR"])

# ============================================================
# 2. CONFIRM EXPECTED FILES ARE PRESENT
# ============================================================
# (Merged in from the old hardcoded-path cell -- this now runs AFTER nb1_dir is actually
# resolved above, instead of racing it with a second, guessed path.)
required_files = [
    "splits.json",
    "A_gauss_maps.h5",
    "config_used.json",
    "patch_meta.csv",
]
print()
for name in required_files:
    path = nb1_dir / name
    print(f"{'\u2713' if path.exists() else '\u2717'} {name}")

# ============================================================
# 3. LOAD NB1 SPLITS
# ============================================================

splits_path = nb1_dir / "splits.json"

if not splits_path.exists():
    raise FileNotFoundError(f"NB1 splits file does not exist:\n{splits_path}")

with open(splits_path, "r") as f:
    splits = json.load(f)

train_idx_full = np.array(splits["train_subsample_idx"])
train_labels_full = np.array(splits["train_subsample_labels"])

print(f"\n\u2713 NB1 train subsample: {len(train_idx_full):,} patches")


def find_pcam_files(root):
    root = Path(root)
    found = {}

    image_names = {
        "train_x": "training_split.h5",
        "valid_x": "validation_split.h5",
        "test_x": "test_split.h5",
    }
    label_names = {
        "train_y": "camelyonpatch_level_2_split_train_y.h5",
        "valid_y": "camelyonpatch_level_2_split_valid_y.h5",
        "test_y": "camelyonpatch_level_2_split_test_y.h5",
    }

    all_h5 = list(root.rglob("*.h5"))
    for key, filename in {**image_names, **label_names}.items():
        matches = [f for f in all_h5 if f.name == filename]
        if matches:
            found[key] = matches[0]
    return found


pcam_files = find_pcam_files(CONFIG["PCAM_DIR"])

required = ["train_x", "train_y", "valid_x", "valid_y", "test_x", "test_y"]
missing = [k for k in required if k not in pcam_files]

if missing:
    print("Could not find:", missing)
    print("\nH5 files found:")
    for f in sorted(Path(CONFIG["PCAM_DIR"]).rglob("*.h5")):
        print(" ", f)
else:
    print("Found all PCam files:")
    for k, v in pcam_files.items():
        print(f"  {k}: {v}")


✓ NB1 checkpoint found:
  /kaggle/input/notebooks/claudeisnotclaude/data-setup/topology_checkpoint

✓ splits.json
✓ A_gauss_maps.h5
✓ config_used.json
✓ patch_meta.csv

✓ NB1 train subsample: 50,000 patches
Found all PCam files:
  train_x: /kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/pcam/training_split.h5
  valid_x: /kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/pcam/validation_split.h5
  test_x: /kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/pcam/test_split.h5
  train_y: /kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_train_y.h5
  valid_y: /kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_valid_y.h5
  test_y: /kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_test_y.h5


## 4. MobileViT implementation & shape validation

Build each candidate backbone architecture-only (no pretrained download yet), confirm it accepts PCam's 96×96 patches, and locate the CNN/Transformer boundary automatically — this boundary is exactly where NB4 will attach `TopoGate`, so it's derived here once and reused downstream rather than re-derived per notebook.

In [5]:
def pick_cnn_transformer_boundary(model):
    '''Return the dotted module path of the last pure-CNN block that runs immediately before
    the first attention block (MobileVitBlock), by walking model.stages in order. This is the
    hybrid-architecture injection point described in TopoDistil_bulletproof.md: LIFE gates the
    CNN feature map right before it gets tokenized into the Transformer, not at the raw input.
    Falls back to the last block overall if no MobileVitBlock is found (i.e. a pure-CNN model),
    matching NB2's plain-midpoint behaviour in that degenerate case.'''
    boundary = None
    for stage_idx, stage in enumerate(model.stages):
        for block_idx, block in enumerate(stage):
            if type(block).__name__ == "MobileVitBlock":
                if boundary is None:
                    raise RuntimeError(
                        "First stage is already a MobileVitBlock -- no pure-CNN block precedes it, "
                        "can't place a CNN-side injection point. Inspect model.stages manually.")
                return boundary
            boundary = f"stages.{stage_idx}.{block_idx}"
    return boundary


def build_and_check(backbone_name, img_size, batch_size=2, pretrained=False):
    '''Builds the backbone, runs a dummy forward pass at PCam resolution, and locates the
    CNN/Transformer injection boundary. Raises on any failure -- callers decide what to do.'''
    model = timm.create_model(backbone_name, pretrained=pretrained, num_classes=2)
    model.eval()

    boundary = pick_cnn_transformer_boundary(model)

    feat_holder = {}
    def _hook(m, i, o):
        feat_holder["F"] = o
    handle = model.get_submodule(boundary).register_forward_hook(_hook)

    dummy = torch.randn(batch_size, 3, img_size, img_size)
    with torch.no_grad():
        out = model(dummy)
    handle.remove()

    return {
        "model": model,
        "injection_layer": boundary,
        "logits_shape": tuple(out.shape),
        "feat_shape_at_injection": tuple(feat_holder["F"].shape),
    }


print(f"Validating shapes at IMG_SIZE={CONFIG['IMG_SIZE']} for each candidate backbone:\n")
shape_results = {}
for name in CONFIG["BACKBONE_CANDIDATES"]:
    try:
        r = build_and_check(name, CONFIG["IMG_SIZE"])
        shape_results[name] = r
        print(f"  {name:16s} OK  logits={r['logits_shape']}  "
              f"injection_layer={r['injection_layer']:14s}  feat_at_injection={r['feat_shape_at_injection']}")
        del r["model"]
    except Exception as e:
        shape_results[name] = {"error": str(e)}
        print(f"  {name:16s} FAILED: {e}")

assert all("error" not in r for r in shape_results.values()), (
    "At least one candidate backbone failed the 96x96 shape check -- fix before continuing. "
    "See shape_results for details.")


Validating shapes at IMG_SIZE=96 for each candidate backbone:

  mobilevit_s      OK  logits=(2, 2)  injection_layer=stages.2.0      feat_at_injection=(2, 96, 12, 12)
  mobilevit_xs     OK  logits=(2, 2)  injection_layer=stages.2.0      feat_at_injection=(2, 64, 12, 12)
  mobilevit_xxs    OK  logits=(2, 2)  injection_layer=stages.2.0      feat_at_injection=(2, 48, 12, 12)


## 5. Parameter count

In [6]:
def count_params(backbone_name):
    model = timm.create_model(backbone_name, pretrained=False, num_classes=2)
    return sum(p.numel() for p in model.parameters())

param_counts = {name: count_params(name) for name in CONFIG["BACKBONE_CANDIDATES"]}
# NB2's RepViT backbone, for reference -- not rebuilt here, just the number the doc's
# size-matching argument (Section 1, point 2) is based on.
param_counts["repvit_m1.dist_in1k (NB2, reference)"] = 4_720_000

param_df = pd.DataFrame(
    [(k, v, v / 1e6) for k, v in param_counts.items()],
    columns=["backbone", "params", "params_millions"],
).sort_values("params")
param_df


,backbone,params,params_millions
2,mobilevit_xxs,951666,0.951666
1,mobilevit_xs,1933618,1.933618
3,"repvit_m1.dist_in1k (NB2, reference)",4720000,4.720000
0,mobilevit_s,4938914,4.938914


## 6. FLOPs

Counted via PyTorch's ATen-level `FlopCounterMode` (falls back to `fvcore`, then `thop`, in that order, if the installed torch build doesn't have it). This matters specifically for MobileViT: its attention block computes `Q @ K^T` and `attn @ V` as raw tensor ops inside `forward()` rather than through an `nn.Module` submodule, so a module-hook-based profiler (like `thop` on its own) never sees those matmuls and silently undercounts every `mobilevit_*` candidate's transformer stages -- while counting RepViT, a pure CNN with no such gap, correctly. That's a profiler blind spot in how the FLOPs get *measured*, not a sign the architecture is a poor fit for the task; the compute those attention layers actually do is identical either way. `FlopCounterMode` hooks the dispatcher itself, so it sees every tensor op regardless of whether it's wrapped in a submodule, and needs no architecture-specific handling to do it.

In [7]:
def count_flops_accurate(backbone_name, img_size):
    """Returns {'flops': float, 'method': str, 'is_lower_bound': bool} for one backbone.
    Tries torch's own FlopCounterMode first (ATen-dispatcher level -- sees every tensor op,
    including MobileViT's raw Q@K^T / attn@V matmuls that a module-hook profiler misses).
    Falls back to fvcore, then thop, if the current torch build lacks FlopCounterMode; thop
    is flagged is_lower_bound=True since it undercounts MobileViT specifically (see markdown
    above), so any table built from these results can show which method actually produced
    each row instead of presenting mismatched numbers as directly comparable."""
    model = timm.create_model(backbone_name, pretrained=False, num_classes=2)
    model.eval()
    dummy = torch.randn(1, 3, img_size, img_size)

    try:
        from torch.utils.flop_counter import FlopCounterMode
        counter = FlopCounterMode(model, display=False)
        with counter:
            model(dummy)
        return {"flops": float(counter.get_total_flops()), "method": "torch_flop_counter_mode",
                "is_lower_bound": False}
    except ImportError:
        pass

    try:
        from fvcore.nn import FlopCountAnalysis
        fca = FlopCountAnalysis(model, dummy)
        fca.unsupported_ops_warnings(False)
        fca.uncalled_modules_warnings(False)
        return {"flops": float(fca.total()), "method": "fvcore", "is_lower_bound": False}
    except ImportError:
        pass

    if HAS_THOP:
        flops, _ = thop_profile(model, inputs=(dummy,), verbose=False)
        return {"flops": float(flops), "method": "thop", "is_lower_bound": True}

    return {"flops": None, "method": None, "is_lower_bound": None}


flops_results = {name: count_flops_accurate(name, CONFIG["IMG_SIZE"])
                  for name in CONFIG["BACKBONE_CANDIDATES"]}

flops_df = pd.DataFrame(
    [(k, v["flops"], (v["flops"] / 1e6 if v["flops"] is not None else None),
      v["method"], v["is_lower_bound"])
     for k, v in flops_results.items()],
    columns=["backbone", "flops", "flops_millions", "method", "is_lower_bound"],
).sort_values("flops")

methods_used = flops_df["method"].dropna().unique().tolist()
if any(flops_df["is_lower_bound"].fillna(False)):
    print("NOTE: at least one backbone fell back to thop (is_lower_bound=True in the table "
          "above) -- torch's FlopCounterMode / fvcore were unavailable on this runtime. "
          "Upgrade torch or check the fvcore install if you need a directly-comparable "
          "RepViT-vs-MobileViT FLOPs number.")
elif len(methods_used) == 1:
    print(f"All candidates counted via '{methods_used[0]}' -- directly comparable, "
          f"RepViT-vs-MobileViT FLOPs numbers below are apples-to-apples.")

flops_df


/tmp/ipykernel_22/2397385390.py:15: UserWarning: mods argument is not needed anymore, you can stop passing it
  counter = FlopCounterMode(model, display=False)


All candidates counted via 'torch_flop_counter_mode' -- directly comparable, RepViT-vs-MobileViT FLOPs numbers below are apples-to-apples.


,backbone,flops,flops_millions,method,is_lower_bound
2,mobilevit_xxs,95589632.0,95.589632,torch_flop_counter_mode,False
1,mobilevit_xs,262414464.0,262.414464,torch_flop_counter_mode,False
0,mobilevit_s,531950080.0,531.950080,torch_flop_counter_mode,False


## 7. GPU memory (with automatic fallback)

Tries each candidate backbone in order (largest/most-capacity first) with a real forward + backward pass at `CONFIG["BATCH_SIZE"]`. The first one that fits without a CUDA OOM becomes `CONFIG["BACKBONE"]` for the rest of this notebook — and the one NB4 should use too.

In [8]:
def measure_gpu_memory(backbone_name, batch_size, img_size, pretrained=True):
    '''Runs one real training step (forward + backward + optimizer.step) and returns peak
    allocated CUDA memory in MB. Returns None (with a printed note) if no GPU is available --
    this check is meaningless on CPU, but shouldn't crash the notebook on a CPU-only session.'''
    if DEVICE != "cuda":
        print(f"  [{backbone_name}] skipped -- no GPU in this session.")
        return None

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(DEVICE)

    model = timm.create_model(backbone_name, pretrained=pretrained, num_classes=2).to(DEVICE)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["LR"])

    dummy_img = torch.randn(batch_size, 3, img_size, img_size, device=DEVICE)
    dummy_labels = torch.randint(0, 2, (batch_size,), device=DEVICE)

    logits = model(dummy_img)
    loss = F_.cross_entropy(logits, dummy_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    torch.cuda.synchronize()

    peak_mb = torch.cuda.max_memory_allocated(DEVICE) / 1e6

    del model, optimizer, dummy_img, dummy_labels, logits, loss
    torch.cuda.empty_cache()
    return peak_mb


gpu_memory_results = {}
chosen_backbone = None
for name in CONFIG["BACKBONE_CANDIDATES"]:
    print(f"Trying {name} at batch_size={CONFIG['BATCH_SIZE']}...")
    # Bugfix: previously this branch fell into the try block even with no GPU, and
    # measure_gpu_memory() would return None (without raising) -- causing chosen_backbone to be
    # set and the loop to break on the very FIRST candidate regardless of whether it would
    # actually fit. That made the "if chosen_backbone is None" no-GPU fallback below unreachable
    # dead code, so its warning message never printed. Now we skip the try/except entirely when
    # there's no GPU and let every candidate fall through to the fallback after the loop.
    if DEVICE != "cuda":
        gpu_memory_results[name] = {"status": "skipped_no_gpu", "peak_mb": None}
        print(f"  [{name}] skipped -- no GPU in this session.")
        continue
    try:
        peak_mb = measure_gpu_memory(name, CONFIG["BATCH_SIZE"], CONFIG["IMG_SIZE"])
        gpu_memory_results[name] = {"status": "ok", "peak_mb": peak_mb}
        print(f"  OK -- peak {peak_mb:.0f} MB")
        chosen_backbone = name
        break
    except torch.cuda.OutOfMemoryError as e:
        gpu_memory_results[name] = {"status": "oom", "error": str(e)}
        print(f"  OOM at batch_size={CONFIG['BATCH_SIZE']} -- falling back to the next-smaller candidate.")
        torch.cuda.empty_cache()

if chosen_backbone is None:
    if DEVICE != "cuda":
        # No GPU to test against -- default to the primary candidate and let the real
        # training run in Section 9 be the actual test once a GPU session is used.
        chosen_backbone = CONFIG["BACKBONE_CANDIDATES"][0]
        print(f"\nNo GPU available to probe -- defaulting to {chosen_backbone}. "
              "Re-run this cell on a GPU session before trusting the memory numbers.")
    else:
        raise RuntimeError(
            f"All candidates {CONFIG['BACKBONE_CANDIDATES']} OOM'd at batch_size={CONFIG['BATCH_SIZE']} "
            "on this GPU. Lower CONFIG['BATCH_SIZE'] and re-run this cell.")

CONFIG["BACKBONE"] = chosen_backbone
print(f"\nSelected backbone for this notebook (and recommended for NB4): {CONFIG['BACKBONE']}")
gpu_memory_results


Trying mobilevit_s at batch_size=32...


model.safetensors:   0%|          | 0.00/22.4M [00:00<?, ?B/s]

  OK -- peak 870 MB

Selected backbone for this notebook (and recommended for NB4): mobilevit_s


{'mobilevit_s': {'status': 'ok', 'peak_mb': 870.249984}}

## 8. Finalize the chosen backbone + injection layer

Re-derive the CNN/Transformer boundary specifically for `CONFIG["BACKBONE"]` (the fallback probe above may have picked a different candidate than `BACKBONE_CANDIDATES[0]`), and build the actual pretrained model used for the rest of this notebook.

In [9]:
CONFIG["INJECTION_LAYER"] = shape_results[CONFIG["BACKBONE"]]["injection_layer"]
print("Final backbone:", CONFIG["BACKBONE"])
print("Final injection layer (for NB4's TopoGate):", CONFIG["INJECTION_LAYER"])

model = timm.create_model(CONFIG["BACKBONE"], pretrained=True, num_classes=2).to(DEVICE)
print(f"Loaded {CONFIG['BACKBONE']} with pretrained ImageNet weights, "
      f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M params.")


Final backbone: mobilevit_s
Final injection layer (for NB4's TopoGate): stages.2.0
Loaded mobilevit_s with pretrained ImageNet weights, 4.94M params.


## 9. Dataset

Plain image + label dataset -- no topology maps, no entropy. Mirrors `PCamTopoDataset` from NB2 minus everything topology-related, and reuses the same lazy h5 handle pattern (opened once per worker, not reopened per `__getitem__`).

In [10]:
def augment_img(img, rng):
    k = rng.integers(0, 4)
    flip = rng.integers(0, 2)
    img_t = np.rot90(img, k, axes=(0, 1))
    if flip:
        img_t = np.flip(img_t, axis=1)
    return np.ascontiguousarray(img_t)


class PCamBaselineDataset(Dataset):
    '''No topology -- just PCam images + labels, for the plain backbone sanity run.'''
    def __init__(self, indices, split, labels=None, augment=True, seed=0):
        self.indices = np.asarray(indices)
        self.split = split
        self.labels = labels
        self.augment = augment
        self.seed = seed          # bugfix: keep the base seed so each DataLoader worker can
                                   # re-derive its OWN rng (see worker_init_fn in Section 11)
        self.rng = np.random.default_rng(seed)
        self._h5 = None

    def _images(self):
        if self._h5 is None:
            self._h5 = h5py.File(pcam_files[f"{self.split}_x"], "r")
            self._x_key = list(self._h5.keys())[0]
        return self._h5[self._x_key]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        img = np.array(self._images()[idx]).astype(np.float32) / 255.0
        if self.augment:
            img = augment_img(img, self.rng)
        label = int(self.labels[i]) if self.labels is not None else -1
        img_t = torch.from_numpy(img.transpose(2, 0, 1).copy()).float()
        return img_t, label, idx


## 10. Validation / test subsets

In [11]:
def load_labels(split, n):
    with h5py.File(pcam_files[f"{split}_y"], "r") as f:
        y_key = list(f.keys())[0]
        return np.array(f[y_key]).reshape(-1)[:n]

rng = np.random.default_rng(CONFIG["SEED"])
n_valid_full = splits["n_valid"]
n_test_full = splits["n_test"]

y_valid_full = load_labels("valid", n_valid_full)
y_test_full = load_labels("test", n_test_full)

val_idx = rng.choice(n_valid_full, min(CONFIG["VAL_SUBSET"], n_valid_full), replace=False)
test_idx = rng.choice(n_test_full, min(CONFIG["TEST_SUBSET"], n_test_full), replace=False)
val_labels_sub = y_valid_full[val_idx]
test_labels_sub = y_test_full[test_idx]

if CONFIG["SANITY_TRAIN_SUBSET"] is not None and CONFIG["SANITY_TRAIN_SUBSET"] < len(train_idx_full):
    sanity_choice = rng.choice(len(train_idx_full), CONFIG["SANITY_TRAIN_SUBSET"], replace=False)
    sanity_train_idx = train_idx_full[sanity_choice]
    sanity_train_labels = train_labels_full[sanity_choice]
else:
    sanity_train_idx = train_idx_full
    sanity_train_labels = train_labels_full

print(f"Sanity train subset: {len(sanity_train_idx)} / {len(train_idx_full)} full NB1 train patches")
print(f"Val subset: {len(val_idx)} | Test subset: {len(test_idx)}")


Sanity train subset: 3000 / 50000 full NB1 train patches
Val subset: 2000 | Test subset: 2000


## 11. Sanity training run (resumable)

Short run (`CONFIG["SANITY_EPOCHS"]` epochs, single seed) to prove the training loop is correct and to measure real per-epoch wall-clock time on this GPU. Uses the same manifest-based resume pattern as NB2's ablation battery, so re-running this cell after an interrupted session skips work that's already `"done"` instead of retraining from scratch.

In [ ]:
# ---------------------------------------------------------------------------
# Section 11 fix (same root-cause diagnosis + fix as NB2/NB4): LR schedule
# (warmup + cosine), discriminative LR, per-epoch best-checkpoint selection,
# and a resumable epoch-level checkpoint system. Applied here too because NB4
# treats this sanity run's numbers as a reference point (see NB4 Section 4).
#
# NB3 has no TopoGate / residual projector / align head -- the only
# pretrained-vs-fresh split available is timm's own classifier head (freshly
# reinitialized here because num_classes=2 != the pretrained 1000-way head),
# vs. everything else in the backbone (pretrained). That's the split used
# below for discriminative LR.
# ---------------------------------------------------------------------------

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for img, labels, idx in loader:
        img = img.to(DEVICE)
        logits = model(img)
        probs = F_.softmax(logits, dim=1)[:, 1]
        all_probs.append(probs.cpu().numpy())
        all_labels.append(np.asarray(labels))
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    auc = roc_auc_score(all_labels, all_probs)
    acc = accuracy_score(all_labels, all_probs > 0.5)
    return {"auc": auc, "acc": acc}


# Bugfix: DataLoader workers are forked from the same Dataset object, so each worker's
# self.rng starts with an IDENTICAL state -- with num_workers=2, both workers draw the same
# sequence of "random" augmentation choices independently, silently reducing augmentation
# diversity within an epoch. Reseed each worker's rng deterministically off the base seed so the
# workers diverge instead of mirroring each other.
def _worker_init_fn(worker_id):
    info = torch.utils.data.get_worker_info()
    ds = info.dataset
    ds.rng = np.random.default_rng(ds.seed + worker_id + 1)


def make_optimizer(model):
    """Two param groups: pretrained backbone body at a lower LR, the freshly
    reinitialized classifier head at the full LR. Uses get_classifier(), which
    works generically across whichever candidate backbone was resolved
    (mobilevit_s / xs / xxs), rather than hardcoding a module name."""
    classifier = model.get_classifier()
    classifier_param_ids = {id(p) for p in classifier.parameters()}
    backbone_params = [p for p in model.parameters() if id(p) not in classifier_param_ids]
    head_params = list(classifier.parameters())

    backbone_lr = CONFIG["LR"] * CONFIG["LR_BACKBONE_MULT"]
    param_groups = [
        {"params": backbone_params, "lr": backbone_lr, "initial_lr": backbone_lr, "name": "backbone"},
    ]
    if head_params:
        param_groups.append(
            {"params": head_params, "lr": CONFIG["LR"], "initial_lr": CONFIG["LR"], "name": "head"}
        )
    return torch.optim.Adam(param_groups)


def make_scheduler(optimizer, steps_per_epoch, epochs):
    """1-epoch linear warmup, then cosine decay to MIN_LR_FRAC of peak over the
    remaining epochs. With SANITY_EPOCHS as small as 2, this is mostly "warm up
    for epoch 0, decay through epoch 1" -- still the right shape, just short."""
    warmup_steps = max(1, steps_per_epoch * CONFIG["WARMUP_EPOCHS"])
    total_steps = max(warmup_steps + 1, steps_per_epoch * epochs)
    min_frac = CONFIG["MIN_LR_FRAC"]

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = (step - warmup_steps) / float(total_steps - warmup_steps)
        progress = min(max(progress, 0.0), 1.0)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_frac + (1.0 - min_frac) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ---- checkpoint helpers (atomic writes, full resume state) ----------------

def _rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def _set_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and "cuda" in state:
        torch.cuda.set_rng_state_all(state["cuda"])


def _cpu_state_dict(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def _atomic_torch_save(payload, path):
    """Write to a temp file then rename -- rename is atomic on POSIX, so a
    process killed mid-write can never leave a half-written (corrupt)
    checkpoint sitting at `path`."""
    tmp_path = path + ".tmp"
    torch.save(payload, tmp_path)
    os.replace(tmp_path, path)


def _atomic_json_dump(obj, path):
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp_path, path)


def save_run_checkpoint(path, epoch, model, optimizer, scheduler, best_val_auc,
                         best_state, best_epoch, epoch_times):
    payload = {
        "epoch": epoch,
        "model_state": _cpu_state_dict(model),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_val_auc": best_val_auc,
        "best_state": best_state,
        "best_epoch": best_epoch,
        "epoch_times": epoch_times,
        "rng_state": _rng_state(),
    }
    _atomic_torch_save(payload, path)


def train_sanity_run(seed, resume=True):
    set_seed(seed)
    m = timm.create_model(CONFIG["BACKBONE"], pretrained=True, num_classes=2).to(DEVICE)

    train_ds = PCamBaselineDataset(sanity_train_idx, "train", labels=sanity_train_labels,
                                    augment=True, seed=seed)
    train_loader = DataLoader(train_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=True,
                               num_workers=2, drop_last=True, worker_init_fn=_worker_init_fn)
    val_ds = PCamBaselineDataset(val_idx, "valid", labels=val_labels_sub, augment=False)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=2,
                             worker_init_fn=_worker_init_fn)
    test_ds = PCamBaselineDataset(test_idx, "test", labels=test_labels_sub, augment=False)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=2,
                              worker_init_fn=_worker_init_fn)

    steps_per_epoch = len(train_loader)
    optimizer = make_optimizer(m)
    scheduler = make_scheduler(optimizer, steps_per_epoch, CONFIG["SANITY_EPOCHS"])

    rid = f"mobilevit_baseline_sanity__{CONFIG['BACKBONE']}__seed{seed}"
    ckpt_path = f'{CONFIG["OUT_DIR"]}/{rid}_ckpt.pt'

    start_epoch = 0
    best_val_auc = -1.0
    best_state = None
    best_epoch = None
    epoch_times = []

    if resume and os.path.exists(ckpt_path):
        try:
            ckpt = torch.load(ckpt_path, map_location=DEVICE)
            m.load_state_dict(ckpt["model_state"])
            optimizer.load_state_dict(ckpt["optimizer_state"])
            scheduler.load_state_dict(ckpt["scheduler_state"])
            best_val_auc = ckpt["best_val_auc"]
            best_state = ckpt["best_state"]
            best_epoch = ckpt["best_epoch"]
            epoch_times = ckpt["epoch_times"]
            _set_rng_state(ckpt["rng_state"])
            start_epoch = ckpt["epoch"] + 1
            print(f"[{rid}] resuming from checkpoint at epoch {start_epoch} "
                  f"(best_val_auc so far={best_val_auc:.4f} @ epoch {best_epoch})")
        except Exception as resume_err:
            print(f"[{rid}] WARNING: found a checkpoint but failed to load it "
                  f"({resume_err}) -- restarting this run from epoch 0.")
            start_epoch = 0
            best_val_auc = -1.0
            best_state = None
            best_epoch = None
            epoch_times = []

    for epoch in range(start_epoch, CONFIG["SANITY_EPOCHS"]):
        m.train()
        t0 = time.time()
        epoch_losses = []
        pbar = tqdm(train_loader, desc=f"sanity seed{seed} epoch{epoch}", leave=False)
        for img, labels, idx in pbar:
            img = img.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = m(img)
            loss = F_.cross_entropy(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
            epoch_losses.append(float(loss.detach().cpu()))
            pbar.set_postfix(loss=epoch_losses[-1], lr=optimizer.param_groups[0]["lr"])
        epoch_times.append(time.time() - t0)

        # --- per-epoch val AUC + best-checkpoint tracking ---
        epoch_val = evaluate(m, val_loader)
        is_best = epoch_val["auc"] > best_val_auc
        if is_best:
            best_val_auc = epoch_val["auc"]
            best_epoch = epoch
            best_state = _cpu_state_dict(m)

        print(f"  epoch {epoch}: mean loss {np.mean(epoch_losses):.4f}, "
              f"{epoch_times[-1]:.1f}s ({len(train_loader)} steps), "
              f"val_auc={epoch_val['auc']:.4f} "
              f"{'(new best)' if is_best else f'(best={best_val_auc:.4f} @ epoch {best_epoch})'}")

        try:
            save_run_checkpoint(ckpt_path, epoch, m, optimizer, scheduler,
                                 best_val_auc, best_state, best_epoch, epoch_times)
        except Exception as ckpt_err:
            print(f"[{rid}] WARNING: checkpoint save failed at epoch {epoch}: {ckpt_err}")

    # --- final eval uses the BEST checkpoint, not whichever epoch training ended on ---
    if best_state is not None:
        m.load_state_dict(best_state)
    val_metrics = evaluate(m, val_loader)
    test_metrics = evaluate(m, test_loader)

    n_batches_full_train = len(train_idx_full) // CONFIG["BATCH_SIZE"]
    sec_per_step = np.mean(epoch_times) / max(len(train_loader), 1)
    est_full_epoch_sec = sec_per_step * n_batches_full_train

    try:
        if os.path.exists(ckpt_path):
            os.remove(ckpt_path)
    except Exception as cleanup_err:
        print(f"[{rid}] WARNING: could not remove checkpoint file {ckpt_path}: {cleanup_err}")

    return m, {
        "backbone": CONFIG["BACKBONE"],
        "sanity_epochs": CONFIG["SANITY_EPOCHS"],
        "sanity_train_size": len(sanity_train_idx),
        "epoch_times_sec": epoch_times,
        "best_epoch": best_epoch,
        "best_val_auc_during_training": best_val_auc,
        "val": val_metrics,
        "test": test_metrics,
        "est_full_epoch_sec_at_full_train_size": est_full_epoch_sec,
        "est_full_battery_sec_reference": est_full_epoch_sec * CONFIG["FULL_EPOCHS_REFERENCE"],
        "full_train_size_reference": len(train_idx_full),
    }


manifest_path = f'{CONFIG["OUT_DIR"]}/run_manifest.json'
manifest = json.load(open(manifest_path)) if os.path.exists(manifest_path) else {}
rid = f"mobilevit_baseline_sanity__{CONFIG['BACKBONE']}__seed{CONFIG['SEED']}"

if manifest.get(rid, {}).get("status") == "done":
    print(f"{rid} already completed -- loading saved metrics instead of retraining.")
    with open(f'{CONFIG["OUT_DIR"]}/{rid}_metrics.json') as f:
        sanity_metrics = json.load(f)
else:
    print(f"=== Running {rid} ===")
    t0 = time.time()
    try:
        sanity_model, sanity_metrics = train_sanity_run(CONFIG["SEED"])
        sanity_metrics["elapsed_sec"] = time.time() - t0
        _atomic_json_dump(sanity_metrics, f'{CONFIG["OUT_DIR"]}/{rid}_metrics.json')

        weights_path = f'{CONFIG["OUT_DIR"]}/{rid}_weights.pt'
        tmp_weights_path = weights_path + ".tmp"
        torch.save(sanity_model.state_dict(), tmp_weights_path)
        os.replace(tmp_weights_path, weights_path)

        manifest[rid] = {"status": "done", "elapsed_sec": sanity_metrics["elapsed_sec"]}
        _atomic_json_dump(manifest, manifest_path)
        del sanity_model
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    except Exception as e:
        manifest[rid] = {"status": "failed", "error": str(e)}
        _atomic_json_dump(manifest, manifest_path)
        # Note: train_sanity_run checkpoints after every completed epoch, so
        # re-running this cell resumes from the last completed epoch instead
        # of retraining from scratch.
        raise

print(f"\nval AUC={sanity_metrics['val']['auc']:.3f}  test AUC={sanity_metrics['test']['auc']:.3f}")
print(f"Estimated full-size ({sanity_metrics['full_train_size_reference']} patches) epoch time: "
      f"{sanity_metrics['est_full_epoch_sec_at_full_train_size']/60:.1f} min")
print(f"Estimated time for {CONFIG['FULL_EPOCHS_REFERENCE']} epochs (NB2/NB4's real budget): "
      f"{sanity_metrics['est_full_battery_sec_reference']/3600:.2f} h per run")
sanity_metrics


## 12. Inference timing

Matches NB2's `profile_backbone_only` methodology exactly (same warmup/timing pattern), so the latency and throughput numbers are directly comparable between RepViT and MobileViT.

In [13]:
def profile_backbone_inference(backbone_name, img_size, batch_size=1, n_warmup=5, n_iters=50):
    m = timm.create_model(backbone_name, pretrained=False, num_classes=2).to(DEVICE).eval()
    dummy = torch.randn(batch_size, 3, img_size, img_size).to(DEVICE)

    with torch.no_grad():
        for _ in range(n_warmup):
            m(dummy)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_iters):
            m(dummy)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        elapsed = time.time() - t0

    latency_ms = elapsed / n_iters * 1000
    throughput_img_s = (batch_size * n_iters) / elapsed
    del m
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return {"latency_ms": latency_ms, "throughput_img_s": throughput_img_s}


inference_profile = {
    "single_sample": profile_backbone_inference(CONFIG["BACKBONE"], CONFIG["IMG_SIZE"], batch_size=1),
    "batch": profile_backbone_inference(CONFIG["BACKBONE"], CONFIG["IMG_SIZE"], batch_size=CONFIG["BATCH_SIZE"]),
}
with open(f'{CONFIG["OUT_DIR"]}/inference_profile.json', "w") as f:
    json.dump(inference_profile, f, indent=2)
inference_profile


{'single_sample': {'latency_ms': 8.23516845703125,
  'throughput_img_s': 121.43042430980174},
 'batch': {'latency_ms': 16.9320011138916,
  'throughput_img_s': 1889.9124672125192}}

## 13. Package checkpoint bundle for NB4

Saves the resolved backbone name, injection layer, and shape/param/FLOPs/GPU-memory/timing results so NB4 loads them directly instead of re-running any of the checks above.

In [14]:
summary = {
    "backbone": CONFIG["BACKBONE"],
    "injection_layer": CONFIG["INJECTION_LAYER"],
    "img_size": CONFIG["IMG_SIZE"],
    "param_counts": {k: int(v) for k, v in param_counts.items()},
    "flops": {k: (None if v["flops"] is None else float(v["flops"])) for k, v in flops_results.items()},
    "flops_method": {k: v["method"] for k, v in flops_results.items()},
    "flops_is_lower_bound": {k: v["is_lower_bound"] for k, v in flops_results.items()},
    "gpu_memory_probe": {
        k: (v if v.get("peak_mb") is None else {**v, "peak_mb": float(v["peak_mb"])})
        for k, v in gpu_memory_results.items()
    },
    "sanity_training": sanity_metrics,
    "inference_profile": inference_profile,
}
with open(f'{CONFIG["OUT_DIR"]}/mobilevit_validation_summary.json', "w") as f:
    json.dump(summary, f, indent=2)

with open(f'{CONFIG["OUT_DIR"]}/config_used.json', "w") as f:
    json.dump(CONFIG, f, indent=2)

print("MobileViT validation output:")
for f in sorted(Path(CONFIG["OUT_DIR"]).iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:45s} {size_mb:10.2f} MB")

print()
print(f"Resolved backbone: {CONFIG['BACKBONE']}")
print(f"Resolved injection layer: {CONFIG['INJECTION_LAYER']}")
print()
print("Next: click 'Save Version' on this notebook.")
print("In NB4, add this notebook's output as an input to load 'mobilevit_validation_summary.json'")
print("and 'config_used.json' -- reuse CONFIG['BACKBONE'] and CONFIG['INJECTION_LAYER'] directly")
print("rather than re-deriving them, and reuse NB2's TopoGate / ResidualProjector / build_A_proj")
print("classes unchanged, hooked onto this notebook's injection_layer instead of NB2's.")


MobileViT validation output:
  config_used.json                                    0.00 MB
  inference_profile.json                              0.00 MB
  mobilevit_baseline_sanity__mobilevit_s__seed0_metrics.json       0.00 MB
  mobilevit_baseline_sanity__mobilevit_s__seed0_weights.pt      19.94 MB
  mobilevit_validation_summary.json                   0.00 MB
  run_manifest.json                                   0.00 MB

Resolved backbone: mobilevit_s
Resolved injection layer: stages.2.0

Next: click 'Save Version' on this notebook.
In NB4, add this notebook's output as an input to load 'mobilevit_validation_summary.json'
and 'config_used.json' -- reuse CONFIG['BACKBONE'] and CONFIG['INJECTION_LAYER'] directly
rather than re-deriving them, and reuse NB2's TopoGate / ResidualProjector / build_A_proj
classes unchanged, hooked onto this notebook's injection_layer instead of NB2's.
